In [1]:
# Cell 1 — The remediation agent with banking-specific logic

from datetime import datetime

print("""
╔══════════════════════════════════════════════════════════════╗
║  PROJECT APEX — Phase 3: AI Remediation Agent               ║
║  Objective: Automatically diagnose and recommend fixes       ║
║  for every failed customer record before go-live             ║
╚══════════════════════════════════════════════════════════════╝
""")

# Banking-specific remediation rules
# These come from your 10 years of knowing what fixes what

REMEDIATION_RULES = {
    'kyc_expired': {
        'issue': 'KYC documentation expired',
        'regulatory_ref': 'AMLD6 Article 14 — Customer Due Diligence',
        'remediation': 'Trigger KYC refresh workflow in OneDrive CRM. '
                      'Customer must provide updated ID documents.',
        'owner': 'Compliance Team',
        'sla_days': 30,
        'can_migrate_without_fix': False
    },
    'kyc_failed': {
        'issue': 'KYC verification failed',
        'regulatory_ref': 'AMLD6 Article 14 + FATF Recommendation 10',
        'remediation': 'Escalate to MLRO. Account must be reviewed before '
                      'any migration proceeds. May require account closure.',
        'owner': 'MLRO / Financial Crime Team',
        'sla_days': 5,
        'can_migrate_without_fix': False
    },
    'pep_flag': {
        'issue': 'Politically Exposed Person — enhanced due diligence required',
        'regulatory_ref': 'AMLD5 Article 20 — PEP Enhanced Due Diligence',
        'remediation': 'Senior management sign-off required. '
                      'Enhanced monitoring to be configured in target system.',
        'owner': 'Head of Compliance',
        'sla_days': 10,
        'can_migrate_without_fix': False
    },
    'sanctions_flag': {
        'issue': 'Potential sanctions match detected',
        'regulatory_ref': 'OFAC / EU Sanctions Regulation',
        'remediation': 'IMMEDIATE ESCALATION to Legal and Compliance. '
                      'Account frozen pending investigation. Do not migrate.',
        'owner': 'Legal + MLRO',
        'sla_days': 1,
        'can_migrate_without_fix': False
    },
    'missing_national_id': {
        'issue': 'National ID / Tax reference number missing',
        'regulatory_ref': 'CRS / FATCA reporting requirements',
        'remediation': 'Contact customer to obtain tax reference number. '
                      'Update in SAP BODS staging table before extraction.',
        'owner': 'Data Management Team',
        'sla_days': 14,
        'can_migrate_without_fix': False
    },
    'missing_email': {
        'issue': 'No valid email address on record',
        'regulatory_ref': 'Internal data completeness standard DS-04',
        'remediation': 'Attempt outbound contact via phone. '
                      'Update contact details in source system.',
        'owner': 'Customer Operations',
        'sla_days': 21,
        'can_migrate_without_fix': True
    },
    'encoding_error': {
        'issue': 'Character encoding error in customer name',
        'regulatory_ref': 'Internal data quality standard DQ-07',
        'remediation': 'Run BODS data cleanse transform — UTF-8 conversion. '
                      'Validate output against original source record.',
        'owner': 'Data Engineering Team',
        'sla_days': 3,
        'can_migrate_without_fix': True
    },
    'duplicate_record': {
        'issue': 'Duplicate customer record detected',
        'regulatory_ref': 'Internal deduplication policy MIG-12',
        'remediation': 'Run golden record matching in Syniti ADMM. '
                      'Merge accounts under surviving customer ID. '
                      'Validate account balances net to zero before merge.',
        'owner': 'Data Migration Team',
        'sla_days': 7,
        'can_migrate_without_fix': False
    }
}


class BankingRemediationAgent:
    """
    AI Agent that analyses failed banking records and produces
    a structured remediation plan aligned to regulatory requirements.

    Architecture: Observe → Think → Act loop
    Memory: Maintains full audit trail of decisions
    """

    def __init__(self, programme_name):
        self.programme_name = programme_name
        self.audit_trail = []
        self.cases_processed = 0
        self.remediation_plan = []

    def observe(self, record):
        """Observe: identify all issues present in this record"""
        issues = []

        if record.get('kyc_status') == 'Expired':
            issues.append('kyc_expired')
        if record.get('kyc_status') == 'Failed':
            issues.append('kyc_failed')
        if record.get('pep_flag') == 1:
            issues.append('pep_flag')
        if record.get('sanctions_flag') == 1:
            issues.append('sanctions_flag')
        if record.get('has_national_id') == 0:
            issues.append('missing_national_id')
        if record.get('has_email') == 0:
            issues.append('missing_email')
        if record.get('has_encoding_error') == 1:
            issues.append('encoding_error')
        if record.get('is_duplicate') == 1:
            issues.append('duplicate_record')

        return issues

    def think(self, issues, segment):
        """Think: prioritise issues and determine migration decision"""

        if not issues:
            return 'CLEAR', 'No issues detected — clear for migration', []

        # Check for hard blockers
        hard_blockers = [i for i in issues
                        if not REMEDIATION_RULES[i]['can_migrate_without_fix']]

        soft_warnings = [i for i in issues
                        if REMEDIATION_RULES[i]['can_migrate_without_fix']]

        # Determine overall decision
        if 'sanctions_flag' in issues:
            decision = 'HOLD — LEGAL ESCALATION'
        elif hard_blockers:
            decision = 'BLOCK — REMEDIATION REQUIRED'
        elif soft_warnings:
            decision = 'CONDITIONAL — MIGRATE WITH MONITORING'
        else:
            decision = 'CLEAR'

        # Calculate priority based on segment and issue severity
        if segment == 'Private Banking' or 'sanctions_flag' in issues:
            priority = 'P1 — CRITICAL'
        elif 'kyc_failed' in issues or 'pep_flag' in issues:
            priority = 'P2 — HIGH'
        elif hard_blockers:
            priority = 'P3 — MEDIUM'
        else:
            priority = 'P4 — LOW'

        return decision, priority, hard_blockers + soft_warnings

    def act(self, record, issues, decision, priority):
        """Act: generate structured remediation actions"""

        self.cases_processed += 1

        actions = []
        total_sla = 0

        for issue_key in issues:
            rule = REMEDIATION_RULES[issue_key]
            actions.append({
                'issue': rule['issue'],
                'regulatory_ref': rule['regulatory_ref'],
                'action': rule['remediation'],
                'owner': rule['owner'],
                'sla_days': rule['sla_days'],
                'blocker': not rule['can_migrate_without_fix']
            })
            total_sla = max(total_sla, rule['sla_days'])

        case = {
            'case_number': f'APEX-REM-{self.cases_processed:05d}',
            'customer_id': record['customer_id'],
            'customer_segment': record['customer_segment'],
            'migration_decision': decision,
            'priority': priority,
            'issues_count': len(issues),
            'remediation_actions': actions,
            'estimated_resolution_days': total_sla,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M')
        }

        # Add to audit trail — every decision is recorded
        self.audit_trail.append({
            'case': case['case_number'],
            'customer': record['customer_id'],
            'decision': decision,
            'agent': 'BankingRemediationAgent-v1',
            'timestamp': case['timestamp']
        })

        self.remediation_plan.append(case)
        return case

    def process_batch(self, failed_records_df, max_display=5):
        """Run the full Observe-Think-Act loop on a batch of failed records"""

        print(f"Agent activated — processing {len(failed_records_df):,} failed records")
        print(f"Displaying first {max_display} cases in detail\n")

        for i, (_, record) in enumerate(failed_records_df.iterrows()):

            # OBSERVE
            issues = self.observe(record)

            # THINK
            decision, priority, ordered_issues = self.think(
                issues, record['customer_segment']
            )

            # ACT
            case = self.act(record, issues, decision, priority)

            # Display detail for first N records
            if i < max_display:
                print(f"{'─'*60}")
                print(f"CASE   : {case['case_number']}")
                print(f"CUSTOMER: {record['customer_id']} | "
                      f"{record['customer_segment']}")
                print(f"DECISION: {decision}")
                print(f"PRIORITY: {priority}")
                print(f"ISSUES  : {len(issues)} found")
                for action in case['remediation_actions']:
                    status = '🔴 BLOCKER' if action['blocker'] else '🟡 WARNING'
                    print(f"  {status} {action['issue']}")
                    print(f"    Action : {action['action'][:70]}...")
                    print(f"    Owner  : {action['owner']}")
                    print(f"    SLA    : {action['sla_days']} days")
                    print(f"    Ref    : {action['regulatory_ref']}")
                print(f"Est. Resolution: {case['estimated_resolution_days']} days")

        if len(failed_records_df) > max_display:
            print(f"\n{'─'*60}")
            print(f"... {len(failed_records_df) - max_display:,} additional cases "
                  f"processed (suppressed for display)")

        return self.remediation_plan


# Activate the agent
agent = BankingRemediationAgent("Project APEX — PrimeBank Acquisition")


╔══════════════════════════════════════════════════════════════╗
║  PROJECT APEX — Phase 3: AI Remediation Agent               ║
║  Objective: Automatically diagnose and recommend fixes       ║
║  for every failed customer record before go-live             ║
╚══════════════════════════════════════════════════════════════╝



In [3]:
# Cell 2 — Run the agent on all failed records

import pandas as pd
import numpy as np

# Create a dummy DataFrame 'df' for demonstration purposes
# This assumes 'df' would have been loaded or generated in a previous step.
np.random.seed(42) # for reproducibility
num_records = 1000
data = {
    'customer_id': range(1001, 1001 + num_records),
    'customer_segment': np.random.choice(['Retail', 'SME', 'Private Banking'], num_records),
    'migration_failed': np.random.randint(0, 2, num_records), # 0 for success, 1 for failed
    'kyc_status': np.random.choice(['Clear', 'Expired', 'Failed'], num_records, p=[0.8, 0.1, 0.1]),
    'pep_flag': np.random.randint(0, 2, num_records),
    'sanctions_flag': np.random.randint(0, 2, num_records),
    'has_national_id': np.random.randint(0, 2, num_records),
    'has_email': np.random.randint(0, 2, num_records),
    'has_encoding_error': np.random.randint(0, 2, num_records),
    'is_duplicate': np.random.randint(0, 2, num_records),
}
df = pd.DataFrame(data)


# Rebuild failed records from previous notebook logic
# (same data generation as before)
failed_sample = df[df['migration_failed'] == 1].head(500).copy()
failed_sample = failed_sample.rename(columns={
    'customer_segment': 'customer_segment'
})

# Add missing columns the agent expects
for col in ['has_national_id', 'has_email', 'has_encoding_error',
            'is_duplicate', 'pep_flag', 'sanctions_flag', 'kyc_status']:
    if col not in failed_sample.columns:
        failed_sample[col] = 0

remediation_plan = agent.process_batch(failed_sample, max_display=4)


Agent activated — processing 473 failed records
Displaying first 4 cases in detail

────────────────────────────────────────────────────────────
CASE   : APEX-REM-00001
CUSTOMER: 1005 | Retail
DECISION: HOLD — LEGAL ESCALATION
PRIORITY: P1 — CRITICAL
ISSUES  : 6 found
  🔴 BLOCKER Politically Exposed Person — enhanced due diligence required
    Action : Senior management sign-off required. Enhanced monitoring to be configu...
    Owner  : Head of Compliance
    SLA    : 10 days
    Ref    : AMLD5 Article 20 — PEP Enhanced Due Diligence
  🔴 BLOCKER Potential sanctions match detected
    Action : IMMEDIATE ESCALATION to Legal and Compliance. Account frozen pending i...
    Owner  : Legal + MLRO
    SLA    : 1 days
    Ref    : OFAC / EU Sanctions Regulation
  🔴 BLOCKER National ID / Tax reference number missing
    Action : Contact customer to obtain tax reference number. Update in SAP BODS st...
    Owner  : Data Management Team
    SLA    : 14 days
    Ref    : CRS / FATCA reporting req

In [4]:
# Cell 3 — Remediation summary for steering committee

plan_df = pd.DataFrame([{
    'Case': c['case_number'],
    'Customer_ID': c['customer_id'],
    'Segment': c['customer_segment'],
    'Decision': c['migration_decision'],
    'Priority': c['priority'],
    'Issues': c['issues_count'],
    'Resolution_Days': c['estimated_resolution_days']
} for c in remediation_plan])

print("=" * 65)
print("  REMEDIATION PLAN SUMMARY — PROJECT APEX STEERING COMMITTEE")
print("=" * 65)
print()

decision_summary = plan_df['Decision'].value_counts()
priority_summary = plan_df['Priority'].value_counts()

print("Cases by Migration Decision:")
display(decision_summary.to_frame('Count'))

print("\nCases by Priority:")
display(priority_summary.to_frame('Count'))

print(f"\nAverage estimated resolution: "
      f"{plan_df['Resolution_Days'].mean():.0f} days")
print(f"Maximum resolution time    : "
      f"{plan_df['Resolution_Days'].max()} days")
print()
print("⚠ CRITICAL PATH: Sanctions cases must be resolved within 24 hours")
print("⚠ KYC refresh programme must begin immediately to meet go-live date")

# Audit trail
print(f"\nAudit trail entries recorded: {len(agent.audit_trail)}")
print("All agent decisions are logged for regulatory review")

  REMEDIATION PLAN SUMMARY — PROJECT APEX STEERING COMMITTEE

Cases by Migration Decision:


,Count
Decision,
HOLD — LEGAL ESCALATION,241
BLOCK — REMEDIATION REQUIRED,212
CONDITIONAL — MIGRATE WITH MONITORING,14
CLEAR,6



Cases by Priority:


,Count
Priority,
P1 — CRITICAL,317
P2 — HIGH,81
P3 — MEDIUM,58
P4 — LOW,11
No issues detected — clear for migration,6



Average estimated resolution: 17 days
Maximum resolution time    : 30 days

⚠ CRITICAL PATH: Sanctions cases must be resolved within 24 hours
⚠ KYC refresh programme must begin immediately to meet go-live date

Audit trail entries recorded: 473
All agent decisions are logged for regulatory review
